# CQT_QPU resonator spectroscopy

Readout only (no drive). Runs on the machine where this notebook's kernel runs, and talks to the cluster over the network (`QRC_CLUSTER_IP`, default `192.168.0.80`). Use the repository's `.venv` as kernel.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qibolab import AcquisitionType, AveragingMode, Parameter, Sweeper, create_platform

os.environ["QIBOLAB_PLATFORMS"] = str(Path("platforms").resolve())  # run from the repo root
platform = create_platform("CQT_QPU")
platform.instruments["cluster0"].address

## Sequence and sweep

The MZ native gate: 1 us square probe, amplitude 0.3 (device config).

In [ ]:
QUBIT = "q2"
SPAN, STEP = 40e6, 0.5e6
NSHOTS = 1024

sequence = platform.natives.single_qubit[QUBIT].MZ.create_sequence()
readout = next(e for _, e in sequence if hasattr(e, "probe"))
probe = platform.qubits[QUBIT].probe
center = platform.parameters.configs[probe].frequency
freqs = center + np.arange(-SPAN / 2, SPAN / 2 + STEP / 2, STEP)
sweeper = Sweeper(parameter=Parameter.frequency, values=freqs, channels=[probe])
readout.probe.amplitude, len(freqs)

## Run on the cluster

In [ ]:
platform.connect()
try:
    res = platform.execute(
        [sequence],
        [[sweeper]],
        nshots=NSHOTS,
        acquisition_type=AcquisitionType.INTEGRATION,
        averaging_mode=AveragingMode.CYCLIC,
    )
finally:
    platform.disconnect()

In [ ]:
iq = np.asarray(res[readout.acquisition.id], dtype=float).reshape(-1, 2)
z = iq[:, 0] + 1j * iq[:, 1]

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
ax1.plot(freqs / 1e9, np.abs(z))
ax1.axvline(center / 1e9, color="gray", ls="--", label="configured")
ax1.set_ylabel("|I + iQ| (a.u.)")
ax1.legend()
ax2.plot(freqs / 1e9, np.unwrap(np.angle(z)))
ax2.set_ylabel("phase (rad)")
ax2.set_xlabel("probe frequency (GHz)")
fig.tight_layout()